In [1]:
from Util.Problems import Problem, solution
import Util.math_functions as mathf
import numpy as np


class P021(Problem):
    number = 21
    title = "Amicable Numbers"
    description = """<p>Let $d(n)$ be defined as the sum of proper divisors of $n$ (numbers less than $n$ which divide evenly into $n$).<br/>
If $d(a) = b$ and $d(b) = a$, where $a \\ne b$, then $a$ and $b$ are an amicable pair and each of $a$ and $b$ are called amicable numbers.</p><p>For example, the proper divisors of $220$ are $1, 2, 4, 5, 10, 11, 20, 22, 44, 55$ and $110$; therefore $d(220) = 284$. The proper divisors of $284$ are $1, 2, 4, 71$ and $142$; so $d(284) = 220$.</p><p>Evaluate the sum of all the amicable numbers under $10000$.</p>"""
    upper_limit = 10000

In [2]:
p = P021()
p.describe()

## Problem 21: Amicable Numbers

<p>Let $d(n)$ be defined as the sum of proper divisors of $n$ (numbers less than $n$ which divide evenly into $n$).<br/>
If $d(a) = b$ and $d(b) = a$, where $a \ne b$, then $a$ and $b$ are an amicable pair and each of $a$ and $b$ are called amicable numbers.</p><p>For example, the proper divisors of $220$ are $1, 2, 4, 5, 10, 11, 20, 22, 44, 55$ and $110$; therefore $d(220) = 284$. The proper divisors of $284$ are $1, 2, 4, 71$ and $142$; so $d(284) = 220$.</p><p>Evaluate the sum of all the amicable numbers under $10000$.</p>

### Solution notes
We start by simply looping through all numbers between $2$ and $10000$, calculating the sum of its divisors, and then checking the resulting sum to see if it is an amicable pair.

In [3]:
@solution(P021, first=True, make_fast=True, warmup_args=(P021.upper_limit, ))
def brute_force(upper_limit):
    amicable_numbers = []
    for a in range(2, upper_limit):
        if a in amicable_numbers:
            continue
        a_divisors = mathf.get_divisors(a)
        b = np.sum(a_divisors) - a # The divisors function also returns the number itself
        if b != a:
            b_divisors = mathf.get_divisors(b)
            if np.sum(b_divisors) - b == a:
                amicable_numbers.extend([a,b])
    return sum(amicable_numbers)

In [4]:
p.test_once("brute_force")

31626 found after a separate test in 3.985300 ms by brute_force (first)


To make this slightly faster, we use a numpy array to track all calculated sums and all found pairs. These arrays can be used for lookup to prevent double calculations.

In [5]:
@solution(P021, make_fast=True, warmup_args=(P021.upper_limit, ))
def arrays(upper_limit):
    sums = np.zeros(upper_limit, dtype=np.int_)
    pairs = np.zeros(upper_limit, dtype=np.int_)
    for a in range(2, upper_limit):
        if pairs[a] != 0:
            continue
        if sums[a] != 0:
            b = sums[a]
        else:
            b = np.sum(mathf.get_divisors(a)) - a # The divisors function also returns the number itself
            sums[a] = b
        if b != a:
            if sums[b] != 0:
                if sums[b] == a:
                    pairs[b] = a
                    pairs[a] = b
            else:
                sums[b] = np.sum(mathf.get_divisors(b)) - b
                if sums[b] == a:
                    pairs[b] = a
                    pairs[a] = b
    return sum(pairs)

In [6]:
p.test_once("arrays")

31626 found after a separate test in 2.206700 ms by arrays


A boolean array seems to make lookup of pairs slightly faster

In [7]:
@solution(P021, make_fast=True, warmup_args=(P021.upper_limit, ))
def boolean_array(upper_limit):
    sums = np.zeros(upper_limit, dtype=np.int_)
    pairs = np.zeros(upper_limit, dtype=np.bool_)
    for a in range(2, upper_limit):
        if pairs[a]:
            continue
        if sums[a] != 0:
            b = sums[a]
        else:
            b = np.sum(mathf.get_divisors(a)) - a # The divisors function also returns the number itself
            sums[a] = b
        if b != a:
            if sums[b] == 0:
                sums[b] = np.sum(mathf.get_divisors(b)) - b
            if sums[b] == a:
                pairs[b] = pairs[a] = True
    pair_sum = 0
    for i, pair in enumerate(pairs):
        if pair:
            pair_sum += i
    return pair_sum

In [8]:
p.test_once("boolean_array")

31626 found after a separate test in 2.292700 ms by boolean_array


Keeping track of a running sum does not seem to make this faster

In [9]:
@solution(P021, best= True, make_fast=True, warmup_args=(P021.upper_limit, ))
def boolean_array_running_sum(upper_limit):
    sums = np.zeros(upper_limit, dtype=np.int_)
    pairs = np.zeros(upper_limit, dtype=np.bool_)
    pair_sum = 0
    for a in range(2, upper_limit):
        if pairs[a]:
            continue
        if sums[a] != 0:
            b = sums[a]
        else:
            b = np.sum(mathf.get_divisors(a)) - a # The divisors function also returns the number itself
            sums[a] = b
        if b != a:
            if sums[b] == 0:
                sums[b] = np.sum(mathf.get_divisors(b)) - b
            if sums[b] == a:
                pairs[b] = pairs[a] = True
                pair_sum += a + b
    return pair_sum

In [10]:
p.test_all()

31626 found after 1000 tests in 2.131137 ms by arrays
31626 found after 1000 tests in 2.032544 ms by boolean_array
31626 found after 1000 tests in 2.045310 ms by boolean_array_running_sum (best)
31626 found after 1000 tests in 3.575881 ms by brute_force (first)
